# Session 3 — Track B: Multi-Agent Orchestration (Instructor Capstone)

In this capstone, students build a **multi-agent system** that decomposes complex tasks, delegates to specialized workers, and orchestrates their collaboration using LangGraph.

> **Instructor Note:** This notebook contains fully solved versions of all 4 milestones with detailed approach explanations. The self-correction loop (Milestone 4) is the most challenging — allow extra time for it.

## Capstone Objectives

By the end of this capstone, students will have built:

1. A **supervisor agent** that decomposes tasks and routes to workers
2. **Specialized worker agents** with distinct tools and capabilities
3. A **LangGraph orchestration** layer managing agent collaboration
4. An **evaluation and self-correction** loop for quality assurance

In [ ]:
# ============================================================
# Imports and Setup
# ============================================================

import openai
import json
import time
from typing import TypedDict, Annotated, List, Dict, Optional
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
import operator

print("All imports successful!")

---
## Milestone 1: Supervisor Agent & Task Decomposition

Build a supervisor agent that decomposes complex requests into subtasks and assigns them to workers.

> **Approach:** The supervisor uses a structured prompt that instructs the LLM to return JSON with subtask decomposition. Each subtask includes an ID, description, assigned worker, and dependencies on other subtask IDs. This enables dependency-aware execution in Milestone 3.

In [ ]:
# Milestone 1 — SOLUTION: Supervisor Agent & Task Decomposition

class SupervisorAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.system_prompt = """You are a project supervisor. Decompose the user's request into 2-4 subtasks.

Available workers:
- researcher: Finds and summarizes information on a topic
- writer: Creates content from outlines or research notes
- coder: Writes and explains code snippets
- reviewer: Evaluates quality and suggests improvements

Return ONLY valid JSON in this format:
{
  "subtasks": [
    {"id": 1, "description": "...", "assigned_worker": "researcher", "dependencies": []},
    {"id": 2, "description": "...", "assigned_worker": "writer", "dependencies": [1]}
  ]
}"""

    def decompose(self, request: str) -> Dict:
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=request)
        ])
        try:
            plan = json.loads(response.content)
        except:
            # Fallback: simple two-task plan
            plan = {"subtasks": [
                {"id": 1, "description": f"Research: {request}", "assigned_worker": "researcher", "dependencies": []},
                {"id": 2, "description": f"Write: {request}", "assigned_worker": "writer", "dependencies": [1]}
            ]}
        plan["original_request"] = request
        return plan


# Test
supervisor = SupervisorAgent()
plan = supervisor.decompose("Write a technical blog post about RAG with code examples and a review")
print(json.dumps(plan, indent=2))

---
## Milestone 2: Specialized Worker Agents with Tools

Build specialized worker agents, each with unique system prompts and capabilities.

> **Approach:** Each worker has a specialized system prompt that defines its role and output format. Workers receive both their subtask description and context from previously completed tasks, allowing them to build on each other's work.

In [ ]:
# Milestone 2 — SOLUTION: Specialized Worker Agents

class ResearcherAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.system_prompt = """You are a research specialist. Given a research task:
1. Identify the key topics to cover
2. Provide a structured summary with bullet points
3. Include relevant technical details and terminology
Be thorough but concise."""
    
    def execute(self, subtask, context=""):
        prompt = f"Task: {subtask}"
        if context:
            prompt += f"\n\nContext from previous work:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "researcher"}


class WriterAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
        self.system_prompt = """You are a technical writer. Given a writing task and research context:
1. Create clear, well-structured content
2. Use headings, paragraphs, and transitions
3. Make complex topics accessible
Write in a professional but approachable tone."""
    
    def execute(self, subtask, context=""):
        prompt = f"Task: {subtask}"
        if context:
            prompt += f"\n\nContext from previous work:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "writer"}


class CoderAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.system_prompt = """You are a coding specialist. Given a coding task:
1. Write clean, well-commented Python code
2. Include brief explanations of key concepts
3. Follow best practices and include error handling
Return code in markdown code blocks."""
    
    def execute(self, subtask, context=""):
        prompt = f"Task: {subtask}"
        if context:
            prompt += f"\n\nContext from previous work:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "coder"}


class ReviewerAgent:
    def __init__(self):
        self.llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
        self.system_prompt = """You are a quality reviewer. Given content to review:
1. Check for accuracy, completeness, and clarity
2. Identify specific issues and suggest improvements
3. Provide a quality score (1-5) with justification
Be constructive and specific in your feedback."""
    
    def execute(self, subtask, context=""):
        prompt = f"Task: {subtask}"
        if context:
            prompt += f"\n\nContent to review:\n{context}"
        response = self.llm.invoke([
            SystemMessage(content=self.system_prompt),
            HumanMessage(content=prompt)
        ])
        return {"result": response.content, "worker": "reviewer"}


# Test
researcher = ResearcherAgent()
result = researcher.execute("Research the key components of a RAG pipeline")
print(f"Worker: {result['worker']}")
print(f"Result: {result['result'][:300]}...")

---
## Milestone 3: LangGraph Orchestration

Build a LangGraph workflow that orchestrates the supervisor and workers.

> **Approach:** We define a StateGraph with a supervisor node (creates the plan), a dispatcher node (picks the next task and routes it), worker nodes (execute subtasks), and an assembler node (combines all results). The dispatcher uses conditional routing to send each subtask to the correct worker, and loops back for more tasks until all are complete.

In [ ]:
# Milestone 3 — SOLUTION: LangGraph Orchestration

class AgentState(TypedDict):
    request: str
    plan: Dict
    completed_tasks: List[Dict]
    current_task_idx: int
    output: str

# Initialize agents
supervisor = SupervisorAgent()
workers = {
    "researcher": ResearcherAgent(),
    "writer": WriterAgent(),
    "coder": CoderAgent(),
    "reviewer": ReviewerAgent()
}

def supervisor_node(state: AgentState) -> AgentState:
    """Create the task plan."""
    plan = supervisor.decompose(state["request"])
    return {**state, "plan": plan, "completed_tasks": [], "current_task_idx": 0}

def dispatcher_node(state: AgentState) -> AgentState:
    """Determine the next task to execute."""
    return state  # Just pass through — routing logic is in the conditional edge

def worker_node(state: AgentState) -> AgentState:
    """Execute the current subtask with the assigned worker."""
    subtasks = state["plan"]["subtasks"]
    idx = state["current_task_idx"]
    task = subtasks[idx]
    
    # Build context from completed tasks
    context = "\n\n".join([
        f"[{ct['worker']}] {ct['result'][:300]}" for ct in state["completed_tasks"]
    ])
    
    # Execute with the assigned worker
    worker_type = task["assigned_worker"]
    worker = workers.get(worker_type, workers["writer"])
    result = worker.execute(task["description"], context)
    
    completed = state["completed_tasks"] + [{**result, "subtask_id": task["id"]}]
    return {**state, "completed_tasks": completed, "current_task_idx": idx + 1}

def assembler_node(state: AgentState) -> AgentState:
    """Combine all worker outputs into final output."""
    parts = []
    for ct in state["completed_tasks"]:
        parts.append(f"--- [{ct['worker'].upper()}] ---\n{ct['result']}")
    output = "\n\n".join(parts)
    return {**state, "output": output}

def route_dispatcher(state: AgentState) -> str:
    """Route to worker or assembler based on remaining tasks."""
    subtasks = state["plan"].get("subtasks", [])
    if state["current_task_idx"] < len(subtasks):
        return "worker"
    return "assembler"

# Build the graph
graph = StateGraph(AgentState)
graph.add_node("supervisor", supervisor_node)
graph.add_node("dispatcher", dispatcher_node)
graph.add_node("worker", worker_node)
graph.add_node("assembler", assembler_node)

graph.set_entry_point("supervisor")
graph.add_edge("supervisor", "dispatcher")
graph.add_conditional_edges("dispatcher", route_dispatcher, {"worker": "worker", "assembler": "assembler"})
graph.add_edge("worker", "dispatcher")
graph.add_edge("assembler", END)

app = graph.compile()

# Test
result = app.invoke({
    "request": "Write a short technical overview of RAG with a code example",
    "plan": {}, "completed_tasks": [], "current_task_idx": 0, "output": ""
})

print(f"Completed {len(result['completed_tasks'])} subtasks")
print(f"\nFinal output preview:\n{result['output'][:500]}...")

---
## Milestone 4: Evaluation & Self-Correction Loop

Add evaluation and self-correction to the workflow.

> **Approach:** We add an evaluator node that scores the output on completeness, coherence, and accuracy. If the average score is below 3.5, a reviser node identifies issues and rewrites. We track iteration count and cap retries at 2 to prevent infinite loops.

In [ ]:
# Milestone 4 — SOLUTION: Evaluation & Self-Correction

class EnhancedState(TypedDict):
    request: str
    plan: Dict
    completed_tasks: List[Dict]
    current_task_idx: int
    output: str
    scores: List[Dict]
    revision_count: int

eval_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

def evaluator_node(state: EnhancedState) -> EnhancedState:
    """Evaluate the quality of the assembled output."""
    metrics = {}
    for metric, prompt in {
        "completeness": f"Rate 1-5: Does this output fully address the request?\nRequest: {state['request']}\nOutput: {state['output'][:1000]}",
        "coherence": f"Rate 1-5: Is this output well-organized and easy to follow?\nOutput: {state['output'][:1000]}",
        "accuracy": f"Rate 1-5: Is this output technically accurate?\nOutput: {state['output'][:1000]}"
    }.items():
        resp = eval_llm.invoke([
            SystemMessage(content="Return ONLY a number 1-5."),
            HumanMessage(content=prompt)
        ])
        try:
            metrics[metric] = int(resp.content.strip())
        except:
            metrics[metric] = 3
    
    metrics["average"] = round(sum(metrics.values()) / len(metrics), 2)
    scores = state.get("scores", []) + [metrics]
    print(f"  Evaluation (iteration {len(scores)}): {metrics}")
    return {**state, "scores": scores}

def reviser_node(state: EnhancedState) -> EnhancedState:
    """Revise the output based on evaluation feedback."""
    latest_scores = state["scores"][-1]
    
    # Identify weakest area
    weakest = min(["completeness", "coherence", "accuracy"], key=lambda k: latest_scores[k])
    
    response = eval_llm.invoke([
        SystemMessage(content=f"""Improve this content. Focus especially on {weakest}.
Keep what works well, fix what doesn't. Return the improved version."""),
        HumanMessage(content=f"Original request: {state['request']}\n\nCurrent output:\n{state['output']}")
    ])
    
    revision_count = state.get("revision_count", 0) + 1
    print(f"  Revised (focus: {weakest}, iteration {revision_count})")
    return {**state, "output": response.content, "revision_count": revision_count}

def route_evaluator(state: EnhancedState) -> str:
    """Route based on quality score and revision count."""
    scores = state.get("scores", [])
    revision_count = state.get("revision_count", 0)
    
    if not scores:
        return "end"
    
    latest = scores[-1]["average"]
    if latest >= 3.5 or revision_count >= 2:
        return "end"
    return "revise"

# Build enhanced graph
enhanced_graph = StateGraph(EnhancedState)
enhanced_graph.add_node("supervisor", supervisor_node)
enhanced_graph.add_node("dispatcher", dispatcher_node)
enhanced_graph.add_node("worker", worker_node)
enhanced_graph.add_node("assembler", assembler_node)
enhanced_graph.add_node("evaluator", evaluator_node)
enhanced_graph.add_node("reviser", reviser_node)

enhanced_graph.set_entry_point("supervisor")
enhanced_graph.add_edge("supervisor", "dispatcher")
enhanced_graph.add_conditional_edges("dispatcher", route_dispatcher, {"worker": "worker", "assembler": "assembler"})
enhanced_graph.add_edge("worker", "dispatcher")
enhanced_graph.add_edge("assembler", "evaluator")
enhanced_graph.add_conditional_edges("evaluator", route_evaluator, {"revise": "reviser", "end": END})
enhanced_graph.add_edge("reviser", "evaluator")

enhanced_app = enhanced_graph.compile()

# Test
result = enhanced_app.invoke({
    "request": "Write a concise technical guide on building a RAG pipeline with Python code",
    "plan": {}, "completed_tasks": [], "current_task_idx": 0,
    "output": "", "scores": [], "revision_count": 0
})

print(f"\nCompleted {len(result['completed_tasks'])} subtasks, {result['revision_count']} revisions")
print(f"Score progression: {[s['average'] for s in result['scores']]}")
print(f"\nFinal output:\n{result['output'][:600]}...")

---
## Capstone Summary

Students built a **multi-agent orchestration system** with:

1. **Supervisor** -- Task decomposition and worker assignment
2. **Workers** -- Specialized agents (researcher, writer, coder, reviewer)
3. **Orchestration** -- LangGraph workflow with conditional routing
4. **Self-Correction** -- Quality evaluation with automated revision

**Up Next:** Session 4 — Cross-track presentations, governance, and closing.